# Data Visualizations Notebook

Dieses Notebook enthält tägliche und geglättete (Rolling-Window) Plots für verschiedene Metriken basierend auf `final_daily_df`.

In [1]:
import os
import pandas as pd
import plotly.express as px

# DataFrame laden
final_daily_df = pd.read_csv(
    os.path.join("processed", "final_daily_df.csv"),
    parse_dates=["date"]
)
final_daily_df = final_daily_df.sort_values("date").reset_index(drop=True)

# Rolling-Window Größe
WINDOW = 365

## 1. Tweet-Aktivität über die Zeit

In [2]:
# Täglicher Plot: tweet_count über date
fig1 = px.line(
    final_daily_df,
    x="date",
    y="tweet_count",
    title="Tägliche Tweet-Aktivität",
    labels={"date": "Datum", "tweet_count": "Anzahl Tweets"}
)
fig1.update_layout(xaxis_title="Datum", yaxis_title="Anzahl Tweets")
fig1.show()

In [3]:
# Rolling-Window Plot: 7-Tage-Durchschnitt tweet_count
final_daily_df["tweet_count_roll"] = final_daily_df["tweet_count"].rolling(window=WINDOW, min_periods=1).mean()

fig2 = px.line(
    final_daily_df,
    x="date",
    y="tweet_count_roll",
    title=f"{WINDOW}-Tage Rolling-Average der Tweet-Aktivität",
    labels={"date": "Datum", "tweet_count_roll": f"{WINDOW}-Tage Durchschnitt"}
)
fig2.update_layout(xaxis_title="Datum", yaxis_title="Tweets (Rolling-Average)")
fig2.show()

## 2. Sentiment-Entwicklung über die Zeit

In [4]:
# Täglicher Plot für Sentiment: neg, neu, pos
fig3 = px.line(
    final_daily_df,
    x="date",
    y=["neg", "neu", "pos"],
    title="Tägliches Sentiment (neg, neu, pos)",
    labels={"value": "Sentiment-Score", "variable": "Sentiment-Kategorie", "date": "Datum"}
)
fig3.update_layout(xaxis_title="Datum", yaxis_title="Sentiment-Score")
fig3.show()

In [5]:
# Rolling-Window für Sentiment
for col in ["neg", "neu", "pos"]:
    final_daily_df[f"{col}_roll"] = final_daily_df[col].rolling(window=WINDOW, min_periods=1).mean()

fig4 = px.line(
    final_daily_df,
    x="date",
    y=["neg_roll", "neu_roll", "pos_roll"],
    title=f"{WINDOW}-Tage Rolling-Average des Sentiments",
    labels={"value": "Rolling Sentiment-Score", "variable": "Kategorie", "date": "Datum"}
)
fig4.update_layout(xaxis_title="Datum", yaxis_title=f"Sentiment-Score (Rolling {WINDOW} Tage)")
fig4.show()

## 3. Polarisierung über die Zeit

In [6]:
# Prozentualer Anteil polarisierter Tweets pro Tag
final_daily_df["polarization_pct"] = final_daily_df["polarized"] / final_daily_df["nlp_tweet_count"] * 100
# exclude alle 100% Werte
final_daily_df = final_daily_df[final_daily_df["polarization_pct"] < 100].reset_index(drop=True)
fig5 = px.line(
    final_daily_df,
    x="date",
    y="polarization_pct",
    title="Täglicher Anteil polarisierter Tweets (%)",
    labels={"date": "Datum", "polarization_pct": "Polarisation (%)"}
)
fig5.update_layout(xaxis_title="Datum", yaxis_title="Polarisation in %")
fig5.show()

In [7]:
WINDOW_SMALL = 7
# Rolling-Window für Polarisierung
final_daily_df["polarization_pct_roll"] = final_daily_df["polarization_pct"].rolling(window=WINDOW, min_periods=1).mean()

fig6 = px.line(
    final_daily_df,
    x="date",
    y="polarization_pct_roll",
    title=f"{WINDOW}-Tage Rolling-Average: Polarisierung (%)",
    labels={"date": "Datum", "polarization_pct_roll": f"Polarisation (%) Rolling {WINDOW}"}
)
fig6.update_layout(xaxis_title="Datum", yaxis_title=f"Polarisation (%) Rolling {WINDOW} Tage")
fig6.show()

## 4. Ekman-Emotionen über die Zeit

In [8]:
# Tägliche Ekman-Emotionen
emotion_cols = ["anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"]

fig7 = px.line(
    final_daily_df,
    x="date",
    y=emotion_cols,
    title="Tägliche Ekman-Emotionen",
    labels={"value": "Emotion-Score", "variable": "Emotion", "date": "Datum"}
)
fig7.update_layout(xaxis_title="Datum", yaxis_title="Emotion-Score")
fig7.show()

In [9]:
# Rolling-Window für Ekman-Emotionen
for col in emotion_cols:
    final_daily_df[f"{col}_roll"] = final_daily_df[col].rolling(window=WINDOW, min_periods=1).mean()

fig8 = px.line(
    final_daily_df,
    x="date",
    y=[f"{col}_roll" for col in emotion_cols],
    title=f"{WINDOW}-Tage Rolling-Average der Ekman-Emotionen",
    labels={"value": "Rolling Emotion-Score", "variable": "Emotion", "date": "Datum"}
)
fig8.update_layout(xaxis_title="Datum", yaxis_title=f"Emotion-Score (Rolling {WINDOW} Tage)")
fig8.show()

## 5. Big 5 Persönlichkeitsmerkmale über die Zeit

In [10]:
# Tägliche Big 5
big5_cols = ["Extroversion", "Neuroticism", "Agreeableness", "Conscientiousness", "Openness"]

fig9 = px.line(
    final_daily_df,
    x="date",
    y=big5_cols,
    title="Tägliche Big 5-Persönlichkeitswerte",
    labels={"value": "Persönlichkeits-Score", "variable": "Trait", "date": "Datum"}
)
fig9.update_layout(xaxis_title="Datum", yaxis_title="Persönlichkeits-Score")
fig9.show()

In [11]:
# Rolling-Window für Big 5
for col in big5_cols:
    final_daily_df[f"{col}_roll"] = final_daily_df[col].rolling(window=WINDOW, min_periods=1).mean()

fig10 = px.line(
    final_daily_df,
    x="date",
    y=[f"{col}_roll" for col in big5_cols],
    title=f"{WINDOW}-Tage Rolling-Average der Big 5",
    labels={"value": "Rolling Persönlichkeits-Score", "variable": "Trait", "date": "Datum"}
)
fig10.update_layout(xaxis_title="Datum", yaxis_title=f"Persönlichkeits-Score (Rolling {WINDOW} Tage)")
fig10.show()

## 6. Wort-Verteilung über die Zeit

In [12]:
# Tägliche Wort-Häufigkeiten (Beispiele)
word_cols = ["bitcoin", "dogecoin", "crypto", "ethereum"]

fig11 = px.line(
    final_daily_df,
    x="date",
    y=word_cols,
    title="Tägliche Häufigkeit ausgewählter Wörter",
    labels={"value": "Anzahl Nennungen", "variable": "Wort", "date": "Datum"}
)
fig11.update_layout(xaxis_title="Datum", yaxis_title="Anzahl Nennungen")
fig11.show()

In [13]:
# Rolling-Window für Wort-Häufigkeiten
for col in word_cols:
    final_daily_df[f"{col}_roll"] = final_daily_df[col].rolling(window=WINDOW, min_periods=1).mean()

fig12 = px.line(
    final_daily_df,
    x="date",
    y=[f"{col}_roll" for col in word_cols],
    title=f"{WINDOW}-Tage Rolling-Average: Wort-Häufigkeiten",
    labels={"value": "Rolling Anzahl Nennungen", "variable": "Wort", "date": "Datum"}
)
fig12.update_layout(xaxis_title="Datum", yaxis_title=f"Anzahl Nennungen (Rolling {WINDOW} Tage)")
fig12.show()

## 7. Tweet-Themen (Topics) über die Zeit

In [ ]:
# Tägliche Tweet-Themen (Topics)
topic_cols = [
    "arts_&_culture", "business_&_entrepreneurs", "celebrity_&_pop_culture",
    "diaries_&_daily_life", "family", "fashion_&_style", "film_tv_&_video",
    "fitness_&_health", "food_&_dining", "gaming", "learning_&_educational",
    "music", "news_&_social_concern", "other_hobbies", "relationships",
    "science_&_technology", "sports", "travel_&_adventure", "youth_&_student_life"
]

fig13 = px.line(
    final_daily_df,
    x="date",
    y=topic_cols,
    title="Tägliche Tweet-Themen (Topics)",
    labels={"value": "Anzahl Tweets", "variable": "Thema", "date": "Datum"}
)
fig13.update_layout(xaxis_title="Datum", yaxis_title="Anzahl Tweets pro Thema")
fig13.show()

ValueError: All arguments should have the same length. The length of argument `y` is 19, whereas the length of previously-processed arguments ['date'] is 2884

In [ ]:
# Rolling-Window für Tweet-Themen
for col in topic_cols:
    final_daily_df[f"{col}_roll"] = final_daily_df[col].rolling(window=WINDOW, min_periods=1).mean()

fig14 = px.line(
    final_daily_df,
    x="date",
    y=[f"{col}_roll" for col in topic_cols],
    title=f"{WINDOW}-Tage Rolling-Average der Tweet-Themen",
    labels={"value": "Rolling Anzahl Tweets", "variable": "Thema", "date": "Datum"}
)
fig14.update_layout(xaxis_title="Datum", yaxis_title=f"Anzahl Tweets (Rolling {WINDOW} Tage)")
fig14.show()